# Patent Topic Modeling with TF‑IDF + NMF


In [9]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
import re
from pathlib import Path
import numpy as np
import itertools
import subprocess
import tempfile
import os
import pandas as pd
from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity


## Step 1 — Load and Combine Text

In [2]:
# Note this csv is saved to the Drive as well
df = pd.read_csv("../data/patents/v1_core_expansion/core/claims_added/v4_processed.csv")

print(f"Dataset loaded: {df.shape[0]} patents")

# Convert to list for processing
docs = df["claims"].astype(str).tolist()


Dataset loaded: 6031 patents


## Step 2 — Remove Patent + GPU Boilerplate

In [3]:
custom_stopwords = [

    # Claim/legal glue
    "claim","claims","claimed","recited","reciting",
    "wherein","whereby",
    "comprising","comprises","comprised",
    "including","includes","include",
    "consisting","consist","consists",

    #Candidate ranks
    "social","networking","online","profile","profiles","candidate","candidates",
    "job","ranking","ranked","query","queries","search","member","members","service","services",
    "score","scores", "assistant", "automated", "content",
    "user", "client", "interaction",
    "language", "text", "audio",
    "search", "query", "profile",
    "candidate", "job",
    "media", "item",

    #Telecom
    "wireless", "radio",
    "ue", "equipment",
    "user equipment",
    "base station",
    "transmit", "transmitting",
    "transmission",
    "channel",
    "station" ,

    # Structural quantifiers
    "plurality","first","second","third","fourth","fifth",
    "one","two","at","least",

    # Reference words
    "said","such","thereof","therein","thereon",
    "therewith","thereto","thereby","thereafter",
    "corresponding","associated","respective",

    # Boilerplate nouns
    "embodiment","embodiments",
    "aspect","aspects",
    "disclosure","present",
    "example","examples",

    # Functional filler verbs
    "configured","configure","configuring",
    "adapted","adapt",
    "operable","operate","operating",
    "may","can","could",
    "based",
    "perform","performs","performing",
    "receive","receives","receiving",
    "determine","determines","determining",

    # Diagram/figure noise
    "figure","figures","fig","figs",
    "shown","illustrated","illustrates","illustrating",
    "depicted","depicts","depicting",
    "diagram","diagrams","drawing","drawings",
    "chart","charts","schematic","schematics",
    "flowchart","flowcharts",

    # GPU identity (remove to expose substructure)
    "gpu","gpus","graphics","graphic",

    # Very generic nouns
    "method","apparatus","system","device","devices",
]

stop_words = set(ENGLISH_STOP_WORDS).union(set(custom_stopwords))


def clean_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^a-z0-9_\s\-]", " ", s)         # keep alnum/_/- as token-friendly
    s = re.sub(r"\b\d{1,2}\b", " ", s)             # remove standalone 1-2 digit numbers
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_clean"] = df["claims"].map(clean_text)

# Quick sanity check
df[["claims","text_clean"]].head(3)

,claims,text_clean
0,['1 . A method for detection of a video waterm...,a method for detection of a video watermark fr...
1,"['1 . A method for executing software methods,...",a method for executing software methods compri...
2,['1. A method for applying orbital angular mom...,a method for applying orbital angular momentum...


## Step 3 — Vectorize with TF‑IDF

In [4]:
stop_words = set(w.lower() for w in stop_words) # This is likely unnecessary but keeping to be safe

In [5]:

vectorizer = TfidfVectorizer(
    stop_words=list(stop_words),
    ngram_range=(1, 2),
    min_df=10, 
    max_df=0.4, # ignore common terms
    lowercase=True,
    sublinear_tf=True,        
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_]{1,}\b"  # >=2 chars, starts with letter
)

X = vectorizer.fit_transform(df["text_clean"])
feature_names = np.array(vectorizer.get_feature_names_out())

print(f"TF‑IDF matrix shape: {X.shape}  (docs × terms)")
print(f"Vocabulary size: {len(feature_names):,}")


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['base'] not in stop_words.
  warnings.warn(


TF‑IDF matrix shape: (6031, 16120)  (docs × terms)
Vocabulary size: 16,120


## Step 4 — Fit NMF Topic Model

In [6]:
# NMF hyperparameters
n_topics = 26

nmf = NMF(
    n_components=n_topics,
    init="nndsvda",
    random_state=42,
    max_iter=1000,
    alpha_W=0,        # regularization on W (doc-topic)
    alpha_H=0.3        # regularization on H (topic-term) forces topics to be more distinct
)

W = nmf.fit_transform(X)  # doc-topic matrix
H = nmf.components_       # topic-term matrix

print(f"W shape: {W.shape} (docs × topics)")
print(f"H shape: {H.shape} (topics × terms)")
print(f"Reconstruction error: {nmf.reconstruction_err_:.4f}")


W shape: (6031, 26) (docs × topics)
H shape: (26, 16120) (topics × terms)
Reconstruction error: 74.1635


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


## Step 5 — Inspect Topics

In [7]:
def print_top_words(H, feature_names, n_top_words=15):
    for topic_idx, topic in enumerate(H):
        top = np.argsort(topic)[::-1][:n_top_words]
        words = feature_names[top]
        weights = topic[top]
        print(f"Topic {topic_idx:02d}: " + ", ".join([f"{w}" for w in words]))

print_top_words(H, feature_names, n_top_words=20)


Topic 00: computer readable, transitory, non transitory, transitory computer, cause, readable storage, storage, readable medium, storage medium, executed, instructions executed, processors, program, code, medium instructions, executable, computer implemented, instructions cause, cause processor, executed processor
Topic 01: value, according, quantization, threshold, target, number, maximum, step, quantized, calculating, values, range, error, parameter, obtain, equal, obtained, predetermined, minimum, quantizing
Topic 02: simd, instruction, single instruction, multiple data, instruction multiple, single, data simd, multiple, simd instruction, simd instructions, register, operation, simd processor, parallel, lane, result, loop, source, execution, operations
Topic 03: neural, neural network, network, layer, layers, weight, weights, network model, training, activation, input, output, quantized, quantization, layer neural, model, convolution, parameters, convolutional, loss
Topic 04: signal

## Step 6: Checking number of topics, stability across seeds

Implemented based on stability metrics outlined here: https://arxiv.org/pdf/2410.23186v2

In [11]:


# -----------------------------
# CONFIG
# -----------------------------
k_values = [16, 18, 20, 22, 24, 26, 28]
seeds = [0, 1, 2, 3, 4]

results = []

# -----------------------------
# HELPERS (match repo behavior)
# -----------------------------
def row_normalize(A, eps=1e-12):
    A = np.asarray(A)
    return A / (A.sum(axis=1, keepdims=True) + eps)

def cosine(u, v, eps=1e-12):
    u = np.asarray(u); v = np.asarray(v)
    denom = (np.linalg.norm(u) * np.linalg.norm(v)) + eps
    return float(np.dot(u, v) / denom)

def greedy_topic_alignment_cosine(post_words_run, post_words_ref):
    """
    Identical to the repo's greedy alignment logic:

    for each topic in the run (top_idx),
      for each topic in reference (top_idx2),
        compute cosine(run_topic, ref_topic)
      take best ref topic not yet used; store in new_top_ordering[top_idx]

    Returns:
      new_top_ordering: length K, entries are reference-topic indices
    """
    K = post_words_ref.shape[0]
    new_top_ordering = [-1] * K
    max_cos_vals = [-1.0] * K

    for top_idx in range(K):
        for top_idx2 in range(K):
            new_max = cosine(post_words_run[top_idx, :], post_words_ref[top_idx2, :])
            corresponding_idx = top_idx2
            if (new_max > max_cos_vals[top_idx]) and (corresponding_idx not in new_top_ordering):
                max_cos_vals[top_idx] = new_max
                new_top_ordering[top_idx] = corresponding_idx

    # Safety: fill any remaining -1 (should be rare) with unused refs
    used = set([i for i in new_top_ordering if i != -1])
    unused = [i for i in range(K) if i not in used]
    for i in range(K):
        if new_top_ordering[i] == -1:
            new_top_ordering[i] = unused.pop(0)

    return np.array(new_top_ordering, dtype=int)

def omega_psych_via_r(X):
    """
    Calls R psych::omega exactly as in tm_reliab.R:
      suppressMessages(suppressWarnings(omega(data.frame(X), nfactors=1)))
      returns omega.tot and mean(stats$sd) (used for their OmegaSE)

    X is a 2D numpy array: rows are observations, cols are replications (items).
    """
    X = np.asarray(X)
    if X.ndim != 2 or X.shape[1] < 2:
        return (np.nan, np.nan)

    # Write X to a temp CSV without headers (R will read.matrix)
    with tempfile.TemporaryDirectory() as td:
        csv_path = os.path.join(td, "x.csv")
        np.savetxt(csv_path, X, delimiter=",")

        r_code = r"""
        suppressMessages(suppressWarnings(library(psych)))
        x <- as.matrix(read.csv("%s", header=FALSE))
        # match repo: omega(data.frame(do.call("cbind", ...)), nfactors=1)
        # here x already is cbind'ed matrix across replications
        res <- suppressMessages(suppressWarnings(omega(as.data.frame(x), nfactors=1)))
        omega_tot <- res$omega.tot
        # repo uses omega(... )$stats$sd and then mean() across topics later
        # stats$sd is a vector; return its mean as an analogue to their usage
        sd_mean <- mean(res$stats$sd)
        cat(sprintf("%%.17g,%%.17g\n", omega_tot, sd_mean))
        """ % (csv_path.replace("\\", "/"))

        proc = subprocess.run(
            ["Rscript", "-e", r_code],
            capture_output=True,
            text=True
        )
        if proc.returncode != 0:
            raise RuntimeError(
                "R psych::omega call failed.\n"
                f"STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
            )

        out = proc.stdout.strip().splitlines()[-1]
        omega_tot_str, sd_mean_str = out.split(",")
        return (float(omega_tot_str), float(sd_mean_str))

# -----------------------------
# SWEEP
# -----------------------------
for n_topics in k_values:
    print(f"\n===== k = {n_topics} =====")

    reconstruction_errors = []
    redundancy_scores = []

    # Store aligned theta/post_words for each seed
    # theta: (n_docs x K), post_words: (K x n_vocab), both aligned to seed0 run
    thetas = []
    post_words_list = []

    post_words_ref = None

    for seed in seeds:
        nmf = NMF(
            n_components=n_topics,
            init="nndsvda",
            random_state=seed,
            max_iter=500,
            alpha_W=0,
            alpha_H=0.3
        )

        W = nmf.fit_transform(X)         # (n_docs x K)
        H = nmf.components_              # (K x n_vocab)

        reconstruction_errors.append(nmf.reconstruction_err_)

        # Redundancy within run (same as you had, but on normalized topics)
        H_prob = row_normalize(H)
        S = cosine_similarity(H_prob)
        np.fill_diagonal(S, np.nan)
        redundancy_scores.append({
            "max_sim": np.nanmax(S),
            "mean_sim": np.nanmean(S)
        })

        # Construct "theta" and "post_words" analogs consistent with paper/repo
        theta = row_normalize(W)         # docs over topics
        post_words = row_normalize(H)    # topics over words

        # Align to first replication using repo's greedy cosine matching
        if post_words_ref is None:
            post_words_ref = post_words.copy()
            thetas.append(theta)
            post_words_list.append(post_words)
        else:
            new_top_ordering = greedy_topic_alignment_cosine(post_words, post_words_ref)

            # Repo reorders:
            # theta[[rep]] <- theta[[rep]][, new_top_ordering]
            # post_words[[rep]] <- post_words[[rep]][new_top_ordering, ]
            theta_aligned = theta[:, new_top_ordering]
            post_words_aligned = post_words[new_top_ordering, :]

            thetas.append(theta_aligned)
            post_words_list.append(post_words_aligned)

    # -----------------------------
    # STABILITY (IDENTICAL): Multidimensional Omega from tm_reliab.R
    # - compute per-topic omega across replications for theta and post_words
    # - use topics 0..K-2 (drop last) for identifiability, exactly like repo
    # -----------------------------
    K_eff = n_topics - 1  # repo uses 1:(ncol(theta[[1]])-1)

    omega1_vals = []  # theta-side omega.tot per topic
    omega2_vals = []  # post_words-side omega.tot per topic

    omega1_sd_means = []  # mean(stats$sd) per topic (for OmegaSE)
    omega2_sd_means = []

    for t in range(K_eff):
        # theta per topic: rows=docs, cols=replications
        X_theta_t = np.column_stack([thetas[r][:, t] for r in range(len(seeds))])
        o1, sd1 = omega_psych_via_r(X_theta_t)
        omega1_vals.append(o1)
        omega1_sd_means.append(sd1)

        # post_words per topic: rows=words, cols=replications
        X_words_t = np.column_stack([post_words_list[r][t, :] for r in range(len(seeds))])
        o2, sd2 = omega_psych_via_r(X_words_t)
        omega2_vals.append(o2)
        omega2_sd_means.append(sd2)

    omega1 = float(np.mean(omega1_vals))
    omega2 = float(np.mean(omega2_vals))
    omega_val = (omega1 + omega2) / 2.0

    # Repo SE:
    # omega_se <- mean(c(mean(unlist(lapply(... omega1_vals[[x]]$stats$sd))),
    #                    mean(unlist(lapply(... omega2_vals[[x]]$stats$sd))))) * sqrt(1/len(by_col)+1/len(post_words_by_col))
    omega_se = float(
        np.mean([np.mean(omega1_sd_means), np.mean(omega2_sd_means)])
        * np.sqrt((1.0 / K_eff) + (1.0 / K_eff))
    )

    # ---- aggregate redundancy
    avg_max_sim = np.mean([r["max_sim"] for r in redundancy_scores])
    avg_mean_sim = np.mean([r["mean_sim"] for r in redundancy_scores])

    print(f"Avg Reconstruction Error: {np.mean(reconstruction_errors):.4f}")
    print(f"Omega (theta-side):      {omega1:.3f}")
    print(f"Omega (words-side):      {omega2:.3f}")
    print(f"Omega (avg):             {omega_val:.3f}")
    print(f"Omega SE (repo-style):   {omega_se:.4f}")
    print(f"Avg Max Topic Cosine:    {avg_max_sim:.3f}")
    print(f"Avg Mean Topic Cosine:   {avg_mean_sim:.3f}")

    results.append({
        "k": n_topics,
        "recon_error": float(np.mean(reconstruction_errors)),
        "omega_theta": omega1,
        "omega_words": omega2,
        "omega": omega_val,
        "omega_se": omega_se,
        "max_cosine": float(avg_max_sim),
        "mean_cosine": float(avg_mean_sim),
    })

# -----------------------------
# SUMMARY TABLE
# -----------------------------
results_df = pd.DataFrame(results)
print("\n===== SUMMARY =====")
print(results_df.sort_values("k"))


===== k = 16 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve 

Avg Reconstruction Error: 74.7555
Omega (theta-side):      0.998
Omega (words-side):      0.998
Omega (avg):             0.998
Omega SE (repo-style):   0.0002
Avg Max Topic Cosine:    0.291
Avg Mean Topic Cosine:   0.121

===== k = 18 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve 

Avg Reconstruction Error: 74.6350
Omega (theta-side):      0.906
Omega (words-side):      0.906
Omega (avg):             0.906
Omega SE (repo-style):   0.0342
Avg Max Topic Cosine:    0.306
Avg Mean Topic Cosine:   0.112

===== k = 20 =====


...

Avg Reconstruction Error: 74.2786
Omega (theta-side):      0.913
Omega (words-side):      0.919
Omega (avg):             0.916
Omega SE (repo-style):   0.0078
Avg Max Topic Cosine:    0.300
Avg Mean Topic Cosine:   0.091

===== k = 26 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve 

Avg Reconstruction Error: 74.1658
Omega (theta-side):      0.892
Omega (words-side):      0.900
Omega (avg):             0.896
Omega SE (repo-style):   0.0042
Avg Max Topic Cosine:    0.291
Avg Mean Topic Cosine:   0.086

===== k = 28 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve 

Avg Reconstruction Error: 74.0587
Omega (theta-side):      0.839
Omega (words-side):      0.838
Omega (avg):             0.839
Omega SE (repo-style):   0.0113
Avg Max Topic Cosine:    0.286
Avg Mean Topic Cosine:   0.081

===== SUMMARY =====
    k  recon_error  omega_theta  omega_words     omega  omega_se  max_cosine  \
0  16    74.755508     0.997508     0.998189  0.997849  0.000196    0.291197   
1  18    74.634979     0.905535     0.906054  0.905795  0.034153    0.306247   
2  20    74.512550     0.966353     0.967368  0.966861  0.010078    0.315026   
3  22    74.395642     0.913340     0.919200  0.916270  0.016433    0.304924   
4  24    74.278611     0.912971     0.919238  0.916104  0.007804    0.299964   
5  26    74.165769     0.891854     0.900366  0.896110  0.004197    0.291408   
6  28    74.058687     0.839235     0.838109  0.838672  0.011280    0.285823   

   mean_cosine  
0     0.121115  
1     0.111958  
2     0.104694  
3     0.097258  
4     0.090799  
5     0.085692 